# 01 · Découverte du dataset

**SceneSense — A4 / Deep Learning.** Objectif : inspecter les données et enregistrer une séparation fixe. Les images du test ne participent pas à l'entraînement.

In [ ]:
from pathlib import Path
import sys
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/SceneSense')]
ROOT = next((p for p in candidates if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the SceneSense project folder in Colab or Jupyter')
sys.path.insert(0, str(ROOT / 'src'))
from scenesense.config import Settings, CLASSES
settings = Settings(data_dir=ROOT / "data/intel-image-classification", output_dir=ROOT / "results")
print("Project:", ROOT)
print("Dataset:", settings.data_dir)

## Construire les trois ensembles

La répartition est stratifiée par catégorie. La partie officielle `seg_test` reste indépendante.

In [ ]:
from scenesense.data import build_manifest, read_manifest
frame = build_manifest(settings.data_dir, settings.manifest_path,
                       settings.seed, settings.validation_fraction)
print(frame.groupby(["split", "label"]).size().unstack(fill_value=0).rename(columns=dict(enumerate(CLASSES))))
print("Saved split:", settings.manifest_path)

## Exemples d'images

Chaque catégorie a des caractéristiques visuelles distinctes, mais les classes glacier et montagne peuvent se ressembler.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, cls in zip(axes.ravel(), CLASSES):
    row = frame[(frame.label == CLASSES.index(cls)) & (frame.split == "train")].iloc[0]
    with Image.open(settings.data_dir / row.path) as im:
        ax.imshow(im.convert("RGB"))
    ax.set_title(cls.capitalize())
    ax.axis("off")
plt.tight_layout()
plt.show()